# Capstone: Compliance Ticket Classifier

In [1]:
!pip install -q "transformers>=4.40" "accelerate>=0.30" wandb scikit-learn
!pip install -q "datasets==2.19.0"

import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 19.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.3.1 which is incompatible.
CUDA available: True
GPU: Tesla T4


In [2]:
from google.colab import userdata
import wandb, os

os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
wandb.login()
print("W&B ready.")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: k-dubas (k-dubas-set-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B ready.


In [3]:
import pandas as pd

train_df = pd.read_parquet("train.parquet")
test_df = pd.read_parquet("test.parquet")

print(f"Train: {len(train_df)} | Test: {len(test_df)}")

# Binary target: 1 = relevant, 0 = not_relevant
train_df["labels"] = (train_df["label"] == "relevant").astype(int)
test_df["labels"] = (test_df["label"] == "relevant").astype(int)

print("\nTrain label balance:")
print(train_df["labels"].value_counts())
print("\nTest label balance (real only):")
print(test_df["labels"].value_counts())

Train: 491 | Test: 132

Train label balance:
labels
0    314
1    177
Name: count, dtype: int64

Test label balance (real only):
labels
0    91
1    41
Name: count, dtype: int64


In [5]:
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LEN,
        padding="max_length",
    )

train_ds = Dataset.from_pandas(train_df[["text", "labels"]])
test_ds = Dataset.from_pandas(test_df[["text", "labels"]])

train_ds = train_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

train_ds = train_ds.remove_columns(["text"])
test_ds = test_ds.remove_columns(["text"])

print("Tokenized.")
print("Columns:", train_ds.column_names)
print("First example keys:", list(train_ds[0].keys()))

Map:   0%|          | 0/491 [00:00<?, ? examples/s]

Map:   0%|          | 0/132 [00:00<?, ? examples/s]

Tokenized.
Columns: ['labels', 'input_ids', 'token_type_ids', 'attention_mask']
First example keys: ['labels', 'input_ids', 'token_type_ids', 'attention_mask']


In [6]:
import numpy as np
import torch
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
)

# Class weights to counter imbalance (weight positive class more heavily).
n_neg = (train_df["labels"] == 0).sum()
n_pos = (train_df["labels"] == 1).sum()
weight_for_pos = n_neg / n_pos
class_weights = torch.tensor([1.0, weight_for_pos], dtype=torch.float)
print(f"Class weights [neg, pos]: {class_weights.tolist()}")

# Custom Trainer that applies the weighted loss.
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(
            weight=class_weights.to(logits.device)
        )
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0),
        "f1": f1_score(labels, preds, zero_division=0),
    }

print("Model and metrics ready.")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Class weights [neg, pos]: [1.0, 1.774011254310608]
Model and metrics ready.


In [7]:
import wandb

training_args = TrainingArguments(
    output_dir="./distilbert-compliance",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="wandb",
    run_name="distilbert-compliance",
    seed=42,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)

wandb.init(project="compliance-ticket-classifier", name="distilbert-compliance",
           config={"model": "distilbert-base-uncased", "max_len": MAX_LEN,
                   "epochs": 4, "weighted_loss": True})

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.590340,0.612144,0.727273,0.580645,0.439024,0.500000
2,0.393099,0.690020,0.765152,0.812500,0.317073,0.456140
3,0.344324,0.674603,0.803030,0.826087,0.463415,0.593750
4,0.284336,0.677912,0.803030,0.826087,0.463415,0.593750


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=124, training_loss=0.3777072794975773, metrics={'train_runtime': 91.3958, 'train_samples_per_second': 21.489, 'train_steps_per_second': 1.357, 'total_flos': 130082985480192.0, 'train_loss': 0.3777072794975773, 'epoch': 4.0})

Let's discover how the synthetic addition affects the results:

In [8]:
# --- ABLATION: train on REAL data only (no synthetics) ---

real_train_df = train_df[train_df["is_synthetic"] == False].copy()
print(f"Real-only train: {len(real_train_df)} "
      f"({(real_train_df['labels']==1).sum()} positives)")

real_train_ds = Dataset.from_pandas(real_train_df[["text", "labels"]])
real_train_ds = real_train_ds.map(tokenize, batched=True)
real_train_ds = real_train_ds.remove_columns(["text"])

model_real = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
)

n_neg_r = (real_train_df["labels"] == 0).sum()
n_pos_r = (real_train_df["labels"] == 1).sum()
weights_real = torch.tensor([1.0, n_neg_r / n_pos_r], dtype=torch.float)
print(f"Real-only class weights: {weights_real.tolist()}")

class WeightedTrainerReal(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fct = torch.nn.CrossEntropyLoss(weight=weights_real.to(outputs.logits.device))
        loss = loss_fct(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

args_real = TrainingArguments(
    output_dir="./distilbert-real-only",
    num_train_epochs=3,          # 3 epochs, as epoch 3 was the plateau before
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=10,
    report_to="wandb",
    run_name="distilbert-real-only",
    seed=42,
)

wandb.init(project="compliance-ticket-classifier", name="distilbert-real-only",
           config={"model": "distilbert", "synthetics": False, "epochs": 3})

trainer_real = WeightedTrainerReal(
    model=model_real, args=args_real,
    train_dataset=real_train_ds, eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)
trainer_real.train()
wandb.finish()

Real-only train: 336 (62 positives)


Map:   0%|          | 0/336 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Real-only class weights: [1.0, 4.4193549156188965]


eval/accuracy,▁▄██
eval/f1,▃▁██
eval/loss,▁█▇▇
eval/precision,▁███
eval/recall,▇▁██
eval/runtime,▃▁▇█
eval/samples_per_second,▆█▂▁
eval/steps_per_second,▆█▂▁
train/epoch,▁▂▂▂▃▃▄▄▅▅▆▆▇▇███
train/global_step,▁▂▂▂▃▃▄▄▅▅▆▆▇▇███
+3,...


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.667776,0.698691,0.727273,0.631579,0.292683,0.400000
2,0.636488,0.687428,0.606061,0.400000,0.536585,0.458333
3,0.564256,0.701680,0.651515,0.448980,0.536585,0.488889


eval/accuracy,█▁▄
eval/f1,▁▆█
eval/loss,▇▁█
eval/precision,█▁▂
eval/recall,▁██
eval/runtime,▄█▁
eval/samples_per_second,▄▁█
eval/steps_per_second,▄▁█
train/epoch,▁▂▂▄▅▅▆███
train/global_step,▁▂▂▄▅▅▆███
+3,...


The synthetics shifted the model toward precision; real-only shifted toward recall.

- Real + synthetic: higher precision (0.83), higher F1 (0.59), higher accuracy, but lower recall (0.46). The synthetics (especially those 40 hard negatives) taught the model to be discriminating, so it makes fewer false alarms. But it also catches fewer positives.

- Real-only: higher recall (0.54: it catches more compliance tickets!) but at a steep precision cost (0.45: nearly half its "relevant" calls are wrong). It's trigger-happy: with only 62 real positives and a 4.4× class weight, it over-predicts "relevant."

Why this happens: the real-only model has just 62 positives and a heavy 4.4× weight, so it aggressively flags things (high recall, low precision). Adding synthetics gave it 155 more examples including hard negatives, which reined in the over-flagging (higher precision) but also made it more conservative on true positives (lower recall).

which model is "better" depends on what a DPO Helpdesk actually needs:
- If we optimize for not missing violations (recall matters most: a missed GDPR trigger is costly), the real-only model is arguably preferable despite lower precision; we'd tolerate false alarms to catch more.
- If we optimize for alert quality / not overwhelming the DPO (precision matters: alert fatigue is real, and your capstone explicitly worries about it), the real+synthetic model wins.

Personally, I'd optimize for not missing violations. But we have too much materials to work with here, so we will use real + synthetic.

In [9]:
# Saving the chosen model (real+synthetic) and register it.
import wandb

SAVE_DIR = "./distilbert-compliance-final"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

run = wandb.init(project="compliance-ticket-classifier", name="register-model",
                 job_type="model-registration")
artifact = wandb.Artifact(
    "distilbert-compliance", type="model",
    metadata={"f1": 0.59, "precision": 0.83, "recall": 0.46,
              "trained_on": "real+synthetic", "base": "distilbert-base-uncased"},
)
artifact.add_dir(SAVE_DIR)
run.log_artifact(artifact)
run.finish()
print("Model saved and registered as a W&B artifact.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

wandb: Adding directory to artifact (distilbert-compliance-final)... Done. 1.9s


Model saved and registered as a W&B artifact.
